# Preprocessing - fusion

In [3]:
import os
import torch
import numpy as np
from math import prod, ceil
import time
from tqdm import tqdm
from transformers import Trainer
from torch.utils.data import default_collate
from torch.utils.data import Subset
from torch import nn
import torch.nn.functional as F
import pickle
import rasterio
import warnings
from rasterio.errors import NotGeoreferencedWarning
warnings.filterwarnings(action='ignore', category=NotGeoreferencedWarning)

from transformers import (
    AutoImageProcessor,
    SegformerForSemanticSegmentation,
    TrainingArguments,
    SegformerConfig,
)

### Utils

In [4]:
def sliding_window_inference(model, image, window=512, stride=512, device="cuda"):
    """
    image: [1, 3, H, W]
    returns: stitched logits [1, C, H, W]
    """

    B, C, H, W = image.shape
    C = model.config.num_labels

    output = torch.zeros((B, C, H, W), device=device)
    counter = torch.zeros((B, 1, H, W), device=device)

    for y in range(0, H, stride):
        for x in range(0, W, stride):

            y1 = min(y + window, H)
            x1 = min(x + window, W)

            patch = image[:, :, y:y1, x:x1]

            # pad if needed
            pad_h = window - patch.shape[2]
            pad_w = window - patch.shape[3]

            if pad_h > 0 or pad_w > 0:
                patch = F.pad(patch, (0, pad_w, 0, pad_h))

            with torch.no_grad():
                logits = model(pixel_values=patch).logits

                logits = F.interpolate(
                        logits,
                        size=(window, window),
                        mode="bilinear",
                        align_corners=False
                    )
            
            logits = logits[:, :, :y1-y, :x1-x]

            output[:, :, y:y1, x:x1] += logits
            counter[:, :, y:y1, x:x1] += 1

    return output / counter

def multiscale_logits(model, image_2048, scales, device="cuda"):
    """
    image_2048: [1, 3, 2048, 2048]
    returns: list of aligned logits [1, C, 2048, 2048] per scale
    """

    logits_per_scale = []

    for s in scales:

        # Resize scene
        H_scaled = int(2048 * s)
        W_scaled = int(2048 * s)

        img_scaled = F.interpolate(
            image_2048,
            size=(H_scaled, W_scaled),
            mode="bilinear",
            align_corners=False
        )

        # Sliding window inference
        logits_scaled = sliding_window_inference(
            model,
            img_scaled,
            window=512,
            stride=512,
            device=device
        )

        # Resize logits back to 2048
        logits_resized = F.interpolate(
            logits_scaled,
            size=(2048, 2048),
            mode="bilinear",
            align_corners=False
        )

        # Normalize logits per scale (important)
        logits_resized = logits_resized / (
            logits_resized.std(dim=(2,3), keepdim=True) + 1e-6
        )

        logits_per_scale.append(logits_resized)

    return logits_per_scale


def get_best_checkpoint(training_dir):
    lst_checkpoints = [x for x in os.listdir(training_dir) if 'checkpoint' in x and not 'last' in x and os.path.isdir(os.path.join(training_dir, x))]

    lst_steps = [int(x.split('-')[-1]) for x in lst_checkpoints]
    best_step = np.max(lst_steps)
    best_checkpoint = [x for x in lst_checkpoints if str(best_step) in x][0]

    return os.path.join(training_dir, best_checkpoint)


def predict_seg(pixel_values, segformer, scales, mode='local'):

    B, H, W, C = pixel_values.shape
    device = pixel_values.device

    scale_descriptors = []
    logits_per_scale = []

    with torch.no_grad():

        for s in scales:

            Hs = int(H * s)
            Ws = int(W * s)

            img_scaled = F.interpolate(
                pixel_values.permute(0,3,1,2),
                size=(Hs, Ws),
                mode="bilinear",
                align_corners=False
            )

            logits_scaled = sliding_window_inference(
                segformer,
                img_scaled,
                window=512,
                stride=512,
                device=device
            )

            logits_resized = F.interpolate(
                logits_scaled,
                size=(H, W),
                mode="bilinear",
                align_corners=False
            )

            logits_resized = logits_resized / (
                logits_resized.std(dim=(2,3), keepdim=True) + 1e-6
            )

            # Store small descriptor only
            desc = F.adaptive_avg_pool2d(logits_resized, 1)
            scale_descriptors.append(desc)

            # Temporarily store logits for fusion
            logits_per_scale.append(logits_resized)

    # Compute weights from descriptors (tiny tensor)
    scale_descriptors = torch.cat(scale_descriptors, dim=1)  # [B, K*C, 1, 1]
    stacked_logits = torch.cat(logits_per_scale, dim=1)

    if mode == 'local':
        return stacked_logits
    elif mode == 'global':
        return scale_descriptors

In [5]:
src_images = r"D:\GitHubProjects\Terranum_repo\LandSlides\segformerlandslides\data\dataset_Bern_2048\tiles\images"
src_model = "../results/training/20260211_172648_segmenter_50_epochs_multi_resolution"
batch_size=4
image_size=2048
num_channels=3
segformer = SegformerForSemanticSegmentation.from_pretrained(
    get_best_checkpoint(src_model),
    num_labels=2,
    ignore_mismatched_sizes=True  # <- Important for custom classes
    )

scales = [1.0, 0.75, 0.5, 0.25]

In [12]:
src_results = os.path.join(os.path.dirname(src_images), 'segpredict')
os.makedirs(src_results, exist_ok=True)
list_images = os.listdir(src_images)
num_batches = ceil(len(list_images) / batch_size)
for batch_num in tqdm(range(num_batches), total=num_batches):
    batch = torch.zeros((batch_size, image_size, image_size, num_channels), dtype=torch.float32)
    for i in range(batch_size):
        img_name = list_images[batch_num * batch_size + i]
        img_arr = rasterio.open(os.path.join(src_images, img_name)).read()[:-1, ...]
        img_arr = img_arr.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])[:, None, None]
        std  = np.array([0.229, 0.224, 0.225])[:, None, None]
        img_arr = (img_arr - mean) / std
        img_arr = torch.from_numpy(img_arr[...]).permute(1,2,0).float()
        # print(img_arr.shape)
        batch[i, ...] = img_arr
    logits = predict_seg(
        pixel_values=batch,
        segformer=segformer,
        scales=scales,
        mode='local'
    )
    for i in range(batch_size):
        img_name = list_images[batch_num * batch_size + i]
        src_file = os.path.join(src_results, os.path.splitext(img_name)[0] + '.pickle')
        with open(src_file, 'wb') as f:
            pickle.dump(logits[i, ...], f)


100%|██████████| 383/383 [1:46:25<00:00, 16.67s/it]
